In [ ]:
"""
Factor: upper wick exhaustion fade

Logic:
    Fades stretched upper-wick days that close away from the high.
"""

from __future__ import annotations


def main(datasources, start_date, end_date):
    import dai
    import numpy as np
    import pandas as pd

    table_name = datasources["bar1m"]

    sql = f"""
WITH minute_bar AS (
            SELECT
                date::DATE::DATETIME AS date,
                date AS minute_time,
                instrument,
                open,
                close,
                volume,
                close * volume AS trade_value,
                CASE
                    WHEN ask_price1 > 0
                     AND bid_price1 > 0
                     AND ask_price1 >= bid_price1
                    THEN (ask_price1 - bid_price1) / NULLIF((ask_price1 + bid_price1) / 2.0, 0)
                    ELSE NULL
                END AS rel_spread,
                CASE
                    WHEN ask_price1 > 0 AND bid_price1 > 0
                    THEN (ask_price1 + bid_price1) / 2.0
                    ELSE close
                END AS mid_price,
                CASE
                    WHEN ask_price1 > 0
                     AND bid_price1 > 0
                     AND (COALESCE(bid_volume1, 0) + COALESCE(ask_volume1, 0)) > 0
                    THEN (
                        ask_price1 * COALESCE(bid_volume1, 0)
                        + bid_price1 * COALESCE(ask_volume1, 0)
                    ) / NULLIF(COALESCE(bid_volume1, 0) + COALESCE(ask_volume1, 0), 0)
                    ELSE NULL
                END AS micro_price,
                (
                    COALESCE(bid_volume1, 0) + COALESCE(bid_volume2, 0)
                    + COALESCE(bid_volume3, 0) + COALESCE(bid_volume4, 0)
                    + COALESCE(bid_volume5, 0) + COALESCE(ask_volume1, 0)
                    + COALESCE(ask_volume2, 0) + COALESCE(ask_volume3, 0)
                    + COALESCE(ask_volume4, 0) + COALESCE(ask_volume5, 0)
                ) AS book_depth,
                (
                    COALESCE(bid_volume1, 0) + COALESCE(bid_volume2, 0)
                    + COALESCE(bid_volume3, 0) + COALESCE(bid_volume4, 0)
                    + COALESCE(bid_volume5, 0)
                ) AS bid_depth,
                (
                    COALESCE(ask_volume1, 0) + COALESCE(ask_volume2, 0)
                    + COALESCE(ask_volume3, 0) + COALESCE(ask_volume4, 0)
                    + COALESCE(ask_volume5, 0)
                ) AS ask_depth,
                (
                    COALESCE(bid_volume1, 0) + COALESCE(bid_volume2, 0)
                    + COALESCE(ask_volume1, 0) + COALESCE(ask_volume2, 0)
                ) AS near_depth,
                (
                    COALESCE(bid_volume3, 0) + COALESCE(bid_volume4, 0)
                    + COALESCE(bid_volume5, 0) + COALESCE(ask_volume3, 0)
                    + COALESCE(ask_volume4, 0) + COALESCE(ask_volume5, 0)
                ) AS far_depth,
                (
                    COALESCE(bid_volume3, 0) + COALESCE(bid_volume4, 0)
                    + COALESCE(bid_volume5, 0)
                ) AS bid_far_depth,
                (
                    COALESCE(ask_volume3, 0) + COALESCE(ask_volume4, 0)
                    + COALESCE(ask_volume5, 0)
                ) AS ask_far_depth
            FROM {table_name}
            WHERE open > 0
              AND close > 0
              AND volume >= 0
        ),
        daily AS (
            SELECT
                date,
                instrument,
                FIRST(open ORDER BY minute_time) AS open_price,
                LAST(close ORDER BY minute_time) AS close_price,
                MAX(close) AS high_price,
                MIN(close) AS low_price,
                SUM(volume) AS total_volume,
                SUM(trade_value) AS total_value,
                SUM(CASE WHEN EXTRACT(HOUR FROM minute_time) >= 14 THEN trade_value ELSE 0 END) AS tail_value,
                SUM(CASE WHEN EXTRACT(HOUR FROM minute_time) >= 14 THEN volume ELSE 0 END) AS tail_volume,
                SUM(CASE WHEN EXTRACT(HOUR FROM minute_time) >= 13 THEN trade_value ELSE 0 END) AS pm_value,
                SUM(CASE WHEN EXTRACT(HOUR FROM minute_time) >= 13 THEN volume ELSE 0 END) AS pm_volume,
                SUM(CASE WHEN EXTRACT(HOUR FROM minute_time) < 11 THEN trade_value ELSE 0 END) AS am_value,
                SUM(CASE WHEN EXTRACT(HOUR FROM minute_time) < 11 THEN volume ELSE 0 END) AS am_volume,
                SUM(CASE WHEN EXTRACT(HOUR FROM minute_time) <= 10 THEN trade_value ELSE 0 END) AS early_value,
                SUM(CASE WHEN EXTRACT(HOUR FROM minute_time) <= 10 THEN volume ELSE 0 END) AS early_volume,
                AVG(rel_spread) AS avg_spread,
                AVG(LOG(1.0 + book_depth)) AS avg_log_depth,
                AVG(1.0 * near_depth / NULLIF(book_depth, 0)) AS near_share,
                AVG(1.0 * far_depth / NULLIF(book_depth, 0)) AS far_share,
                AVG(1.0 * (bid_far_depth - ask_far_depth) / NULLIF(bid_far_depth + ask_far_depth, 0)) AS far_tilt,
                AVG(1.0 * (bid_depth - ask_depth) / NULLIF(book_depth, 0)) AS depth_tilt,
                AVG((micro_price - close) / NULLIF(close, 0)) AS micro_premium,
                AVG((mid_price - close) / NULLIF(close, 0)) AS mid_close_gap,
                COUNT(*) AS n_minutes
            FROM minute_bar
            GROUP BY date, instrument
        )
        SELECT
            date,
            instrument,
            (
                
                -1.0
                * (close_price / open_price - 1.0)
                * (high_price - close_price) / NULLIF(high_price - low_price, 0)
                * (0.5 + COALESCE(near_share, 0))
        
            )
            * CASE
                WHEN n_minutes / 240.0 < 0.3 THEN 0.3
                WHEN n_minutes / 240.0 > 1.0 THEN 1.0
                ELSE n_minutes / 240.0
              END AS raw_factor
        FROM daily
        WHERE open_price > 0
          AND close_price > 0
          AND high_price >= low_price
          AND total_volume > 0
    """

    factor = dai.query(
        sql,
        filters={"date": [start_date, end_date]},
        compression=True,
    ).df()

    factor["date"] = pd.to_datetime(factor["date"]).dt.normalize()
    factor["raw_factor"] = pd.to_numeric(factor["raw_factor"], errors="coerce")
    factor["raw_factor"] = factor["raw_factor"].replace([np.inf, -np.inf], np.nan)
    factor = factor.dropna(subset=["raw_factor"])

    med = factor.groupby("date")["raw_factor"].transform("median")
    mad = (factor["raw_factor"] - med).abs().groupby(factor["date"]).transform("median")
    mad = mad.replace(0, np.nan).fillna(1e-8)
    clipped = factor["raw_factor"].clip(lower=med - 5 * mad, upper=med + 5 * mad)
    mean = clipped.groupby(factor["date"]).transform("mean")
    std = clipped.groupby(factor["date"]).transform("std").replace(0, np.nan).fillna(1e-8)
    factor["factor"] = (clipped - mean) / std
    factor["factor"] = pd.to_numeric(factor["factor"], errors="coerce")

    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
    ).df()
    stk_pool["date"] = pd.to_datetime(stk_pool["date"]).dt.normalize()

    return (
        pd.merge(factor, stk_pool, how="inner", on=["date", "instrument"])
        .dropna(subset=["factor"])
        .sort_values(["date", "instrument"])
        .reset_index(drop=True)
        .loc[:, ["date", "instrument", "factor"]]
    )

